## Web scaping competition for athlete data
---
### searching for csv

In [ ]:
# libraries

# web scraping + selenium specific libraires
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# standard libraries
import csv
import pandas as pd
from datetime import datetime, timedelta
import time, os, glob, random
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parent))

In [3]:
USER_AGENTS = [
    # Chrome 120+ Windows
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36",
]

def get_random_user():
    return random.choice(USER_AGENTS)

In [ ]:
# global variables
WAIT_TIME = 60
USER_AGENT = get_random_user()

# create folder to store competition data
DOWNLOAD_DIR = os.path.abspath(os.path.join(os.getcwd(), "data"))
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

print("Current working directory:", os.getcwd())
print("Download directory:", DOWNLOAD_DIR)
print("Directory exists:", os.path.exists(DOWNLOAD_DIR))

# pulls driver to avoid downloading manualy
options = Options()

# headless driver + reliability
options.add_argument("--headless=new")
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920,1080")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)
options.add_argument(f"--user-agent={USER_AGENT}")

# allowing for downloads early
options.add_experimental_option("prefs", {
    "download.default_directory": DOWNLOAD_DIR,
    "download.prompt_for_download": False,
    "download.directory_upgrade": True,
    "safebrowsing.enabled": True,
})

# creating chrome driver
driver = webdriver.Chrome(options=options)

# remove webdriver flag
driver.execute_script(
    "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
)

wait = WebDriverWait(driver,WAIT_TIME)

# allow headless downloads
driver.execute_cdp_cmd("Page.setDownloadBehavior", {
    "behavior": "allow",
    "downloadPath": DOWNLOAD_DIR
})

Current working directory: c:\Users\josed\projects\webscraping_liftingcast\notebooks
Download directory: c:\Users\josed\projects\webscraping_liftingcast\notebooks\data
Directory exists: True


{}

In [5]:
def wait_for_new_csv(download_dir: str, before_files: set[str], timeout: int = 90) -> str:
    end = time.time() + timeout
    while time.time() < end:
        # still downloading?
        if glob.glob(os.path.join(download_dir, "*.crdownload")):
            time.sleep(0.5)
            continue

        current = set(glob.glob(os.path.join(download_dir, "*.csv")))
        new_files = current - before_files
        if new_files:
            return max(new_files, key=os.path.getmtime)

        time.sleep(0.5)

    raise TimeoutError("CSV download did not complete in time.")

### adding athletes to competition
---



In [6]:
conn = init_db()
df = pd.read_sql_query("SELECT * FROM competitions", conn)
conn.close()

In [7]:
link = df['comp_url'][0]
link

'https://liftingcast.com/meets/m99nrpg1qa1o/results'

In [18]:
link = df['comp_url'][0]

# snapshot existing csv files before starting
before = set(glob.glob(os.path.join(DOWNLOAD_DIR, "*.csv")))

# navigate to page
print(f"getting {link} data.\n", "-" * 20)
driver.get(link)
print("link retrieved")

# click into main export button
print("waiting on export button once found")
export_btn = wait.until(
    EC.element_to_be_clickable((By.CSS_SELECTOR, "button.export-button")))
print("export button found")
export_btn.click()
print("export button clicked")

# wait for modal to appear
print("waiting on modal button once visible")
modal = wait.until(
    EC.visibility_of_element_located((By.CSS_SELECTOR, ".export-results-modal"))
)
print("modal found")

# clicking modal
modal_export = wait.until(
    EC.element_to_be_clickable((
        By.XPATH,
        "//div[contains(@class,'export-results-modal')]//button[normalize-space()='Export' or contains(., 'Export')]"
    ))
)
driver.execute_script("arguments[0].click();", modal_export)

# waiting for csv to download
print("waiting for csv download...")
csv_path = wait_for_new_csv(DOWNLOAD_DIR, before, timeout=120)
print("Downloaded:", csv_path)

# quiting driver
driver.quit()

getting https://liftingcast.com/meets/m8qa2atep5qi/results data.
 --------------------
link retrieved
waiting on export button once found
export button found
export button clicked
waiting on modal button once visible
modal found
waiting for csv download...
Downloaded: c:\Users\josed\projects\powerliftingCompHandler\notebooks\data\100_raw_ironbuilt_classic_full_power_and_single_lifts_ashland_va_awards_results.csv


## webscraping lifting cast elements
---
using HTML elements


In [ ]:
def init_driver():
    USER_AGENTS = [
    # Chrome 120+ Windows
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36",
    ]

    def get_random_user():
        return random.choice(USER_AGENTS)
    
    options = Options()
    options.add_argument("--headless")  # Run in headless mode
    options.add_argument("--disable-gpu")  # Disable GPU acceleration
    options.add_argument("--no-sandbox")  # Bypass OS security model
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)
    options.add_argument("--disable-dev-shm-usage")  # Overcome limited resource problems
    options.add_argument(f"user-agent={get_random_user()}")  # Set a random user agent

    driver = webdriver.Chrome(options=options)

    # remove webdriver flag
    driver.execute_script(
        "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
        )
    return driver

# TODO:
- have user select the athlete
- search for board (can be multiple)
- scrape for matching athlete information
- allow user to refresh using an updated url and athlete in memory to avoid previous steps
- click gear button (has options for more data ex.(forcasted dots) which will help with decision making)
- display the dots differnces between different athletes
